## Percepciones y Prácticas de Seguridad Digital

# Método a elegir: Series temporales (Recomendada para IoTCI, CRIC).
El método de series temporales no se considera metodológicamente correcto ni interpretable aplicar series temporales como ARIMA al dataset. Si bien este se considera de naturaleza transversal (Cross-Sectional): El dataset recopila respuestas a una encuesta donde participaron 12.351 personas distribuidas en 12 países en un único momento estático de recolección.  

No es posible hacer uso de la característica "ID" como tiempo, puesto que esta columna permite ordenar a los participantes mediante un campo numérico consecutivo, por lo tanto este número refleja únicamente el orden de registro de las respuestas mas no una progresión temporal como horas, días o meses.

En este orden de ideas, modelos como ARIMA y la descomposición estacional exigen que las observaciones tengan dependencia temporal secuencial, estacionariedad y estacionalidad real (por ejemplo, patrones semanales o estacionales en tráfico de red) y de aAplicarse sobre el índice de participantes generaría artefactos matemáticos carentes de significado científico.

Dado lo anterior, se opta por aplicar una técnica avanzada de modelado predictivo con el fin de  responder a qué factores determinan el comportamiento protector (q20) o la exposición a incidentes (q7), y por lo tanto se opta por un sistema de clasificación supervisada comparando tres algoritmos: Regresión Logística, Random Forest y Gradient Boosting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carga del dataset principal
df = pd.read_csv('SurveyResponses_DigitalSecurityAroundTheWorld.csv', low_memory=False)
print(f"Dimensiones del dataset: {df.shape}")

# Validación de la pregunta de control q13r11
# El valor esperado en el cuestionario es la segunda opción desde la derecha/abajo (código 4 típicamente)
control_dist = df['q13r11'].value_counts(dropna=False)
print("Distribución de la pregunta de control q13r11:")
print(control_dist)

# Limpieza y conversión de q4r8 (texto libre entre códigos numéricos)
df['q4r8'] = pd.to_numeric(df['q4r8'], errors='coerce')

# Construcción de índices sintéticos clave:
# 1. Índice de Conocimiento/Familiaridad (q6r1 a q6r22)
q6_cols = [f'q6r{i}' for i in range(1, 23)]
df['indice_conocimiento'] = df[q6_cols].mean(axis=1)

# 2. Índice de Comportamiento Protector (q20r1 a q20r14)
q20_cols = [f'q20r{i}' for i in range(1, 15)]
df['indice_proteccion'] = df[q20_cols].sum(axis=1)

print(" Índices sintéticos calculados correctamente.")

In [ ]:
# Verificación formal de ausencia de columnas temporales
temporal_checks = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]
print(f"Columnas con componentes temporales detectadas: {temporal_checks}")
print("""
INFORME METODOLÓGICO - PARTE 2.
Técnica evaluada inicialmente: Descomposición estacional y modelos ARIMA (statsmodels.tsa).
Resultado de la auditoría: El dataset es transversal (cross-sectional) y carece de eje temporal.
Conclusión: Se descarta ARIMA por incompatibilidad estructural y se activa el protocolo de 
alternativa avanzada (Evaluación Predictiva con Clasificadores de Machine Learning).
""")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Definición de variable objetivo binaria: Usuario con alta protección (por encima de la mediana)
mediana_prot = df['indice_proteccion'].median()
df['alta_proteccion'] = (df['indice_proteccion'] > mediana_prot).astype(int)

# Selección de características predictoras (Demografía, Conocimiento, Preocupación)
features = ['Age', 'Education', 'indice_conocimiento', 'indice_proteccion']
model_df = df[features + ['alta_proteccion']].dropna()

X = model_df[features]
y = model_df['alta_proteccion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenamiento y evaluación de tres clasificadores
classifiers = {
    "Regresión Logística": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {"AUC-ROC": auc}
    print(f"--- {name} ---")
    print(f"AUC-ROC: {auc:.4f}")
    print(classification_report(y_test, y_pred))

## Modelado Predictivo Diferencial (Random Forest por Bloque Cultural)
Para evaluar si los predictores de la adopción de medidas de protección difieren entre ambos bloques, se entrenaron modelos independientes de Random Forest evaluando el poder predictivo (AUC-ROC) a partir de la edad, la educación y el índice de conocimiento técnico.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

# 1. Carga y definición de bloques culturales WEIRD vs Non-WEIRD
df = pd.read_csv('SurveyResponses_DigitalSecurityAroundTheWorld.csv', low_memory=False)
weird_countries = [2, 4, 5, 7, 10, 11, 12] # Alemania, Israel, Italia, Polonia, Suecia, UK, USA
df['is_weird'] = df['Country'].apply(lambda x: 1 if x in weird_countries else 0)

# 2. Cálculo de índices
q6_cols = [f'q6r{i}' for i in range(1, 23)]
q20_cols = [f'q20r{i}' for i in range(1, 15)]
df['indice_conocimiento'] = df[q6_cols].mean(axis=1)
df['indice_proteccion'] = df[q20_cols].sum(axis=1)

# 3. Evaluación comparativa de modelos predictivos
features = ['Age', 'Education', 'indice_conocimiento']

for label, val in [("WEIRD", 1), ("Non-WEIRD", 0)]:
    subset = df[df['is_weird'] == val][features + ['indice_proteccion']].dropna()
    median_prot = subset['indice_proteccion'].median()
    
    y = (subset['indice_proteccion'] > median_prot).astype(int)
    X = subset[features]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    
    y_prob = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    
    print(f"\n================ NARRATIVA ANALÍTICA: {label} ================")
    print(f"AUC-ROC del Modelo Predictivo: {auc:.4f}")
    print("Importancia relativa de variables:", dict(zip(features, np.round(clf.feature_importances_, 4))))